# Pilot-модель LightGBM: воспроизводимый pipeline

В ноутбуке используется frozen feature set и зафиксированная конфигурация модели. Отбор признаков, tuning и использование target-derived признаков не выполняются.

## Environment & Data Setup

Настроим пути для локальной среды или Google Colab, загрузим разреженные hourly labels и submission template. Raw-файлы остаются неизменными.

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/MT-Hackathon-2026/data')
else:
    DATA_ROOT = Path('../../data').resolve()

RAW_DIR = DATA_ROOT / 'raw'
LABELS_DIR = RAW_DIR / 'labels'
LABELS_TRAIN_PATH = LABELS_DIR / 'labels_day_train.csv'
LABELS_TEST_PATH = LABELS_DIR / 'labels_day_test.csv'
SUBMISSION_TEMPLATE_PATH = RAW_DIR / 'test_submission.csv'
SUBMISSION_PATH = DATA_ROOT / 'processed' / 'pilot_model_v1_submission.csv'

ROUTES = (1, 5, 7, 11, 12, 17, 25, 26, 28, 50)
FEATURES = ('route', 'weekday', 'hour', 'route_hour', 'hour_weekend', 'is_night')
CATEGORICAL_FEATURES = ('route', 'weekday', 'hour', 'route_hour', 'hour_weekend')
MODEL_PARAMS = {
    'objective': 'regression_l1',
    'n_estimators': 300,
    'max_depth': 6,
    'num_leaves': 31,
    'learning_rate': 0.1,
    'random_state': 42,
    'n_jobs': 4,
    'verbosity': -1,
}

for path in (LABELS_TRAIN_PATH, LABELS_TEST_PATH, SUBMISSION_TEMPLATE_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

labels_train_sparse = pd.read_csv(LABELS_TRAIN_PATH, sep=';', parse_dates=['date'])
labels_test_sparse = pd.read_csv(LABELS_TEST_PATH, sep=';', parse_dates=['date'])
submission_template = pd.read_csv(
    SUBMISSION_TEMPLATE_PATH, sep=';', parse_dates=['date']
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Data Preparation

Labels содержат только положительные наблюдавшиеся ячейки. Для исторических периодов строится полный grid `(route, date, hour)`, отсутствующие сочетания интерпретируются как нулевые посадки согласно контракту данных. Сентябрь–октябрь остаются скрытыми от обучения соответствующего fold.

In [15]:
KEYS = ['route', 'date', 'hour']

def make_hourly_grid(start, end, sparse_labels):
    dates = pd.date_range(start=start, end=end, freq='D')
    grid = pd.MultiIndex.from_product(
        [ROUTES, dates, range(24)], names=KEYS
    ).to_frame(index=False)
    observed = sparse_labels.loc[
        sparse_labels['date'].between(pd.Timestamp(start), pd.Timestamp(end)),
        KEYS + ['boardings'],
    ].copy()
    if observed.duplicated(KEYS).any():
        raise ValueError('Duplicate historical label keys')
    hourly = grid.merge(observed, on=KEYS, how='left', validate='one_to_one')
    hourly['boardings'] = hourly['boardings'].fillna(0).astype('int64')
    return hourly

train_history = make_hourly_grid(
    '2025-01-01', '2025-08-31', labels_train_sparse
)
validation_history = make_hourly_grid(
    '2025-09-01', '2025-10-31', labels_test_sparse
)
history = pd.concat([train_history, validation_history], ignore_index=True)
if history.duplicated(KEYS).any() or history['boardings'].isna().any():
    raise ValueError('Historical grid keys or target are invalid')
if set(history['route'].unique()) != set(ROUTES):
    raise ValueError('Historical route set differs from frozen route contract')
display(pd.DataFrame({
    'period': ['Jan-Aug', 'Sep-Oct', 'Jan-Oct'],
    'rows': [len(train_history), len(validation_history), len(history)],
}))

,period,rows
0,Jan-Aug,58320
1,Sep-Oct,14640
2,Jan-Oct,72960


## Feature Engineering

Функции признаков и типы категорий перенесены из frozen-секции `feature_engineering.ipynb`. Порядок колонок фиксирован; дополнительные преобразования и признаки не добавляются.

In [16]:
def make_features(rows):
    weekday = rows['date'].dt.dayofweek
    features = pd.DataFrame(index=rows.index)
    features['route'] = pd.Categorical(rows['route'], categories=ROUTES)
    features['weekday'] = pd.Categorical(weekday, categories=range(7))
    features['hour'] = pd.Categorical(rows['hour'], categories=range(24))
    route_hour_categories = [
        route * 24 + hour for route in ROUTES for hour in range(24)
    ]
    features['route_hour'] = pd.Categorical(
        rows['route'] * 24 + rows['hour'], categories=route_hour_categories
    )
    features['hour_weekend'] = pd.Categorical(
        rows['hour'] * 2 + weekday.ge(5).astype('int8'), categories=range(48)
    )
    features['is_night'] = rows['hour'].between(0, 5).astype('int8')
    return features.loc[:, list(FEATURES)]

assert tuple(make_features(history.head()).columns) == FEATURES

## Validation Strategy

Единица прогноза — `(route, date, hour)`, target — `boardings`. Используются только календарные признаки, известные заранее. WAPE рассчитывается как сумма абсолютных ошибок, делённая на сумму фактических посадок; WAPE-score равен `max(0, 1 - WAPE)`. Для маршрута с нулевой суммой target WAPE не определён и будет показан как `NaN`.

## Time-aware Cross-Validation

Три expanding-window fold охватывают январь–октябрь 2025. Каждый train строго заканчивается до начала своего validation-периода. Метрики fold и WAPE по маршрутам появятся после выполнения ячеек; заранее результаты не фиксируются.

In [17]:
FOLDS = (
    ('2025-01-01', '2025-04-30', '2025-05-01', '2025-06-30'),
    ('2025-01-01', '2025-06-30', '2025-07-01', '2025-08-31'),
    ('2025-01-01', '2025-08-31', '2025-09-01', '2025-10-31'),
)

def make_model():
    return LGBMRegressor(**MODEL_PARAMS)

def postprocess_predictions(raw_predictions):
    return np.floor(np.maximum(raw_predictions, 0) + 0.5).astype('int64')

def calculate_metrics(actual, prediction, routes):
    actual = np.asarray(actual)
    prediction = np.asarray(prediction)
    total_actual = actual.sum()
    if total_actual <= 0:
        raise ValueError('Overall WAPE is undefined for zero total actual')
    wape = np.abs(actual - prediction).sum() / total_actual
    route_frame = pd.DataFrame({
        'route': np.asarray(routes),
        'actual': actual,
        'absolute_error': np.abs(actual - prediction),
    })
    route_metrics = route_frame.groupby('route').agg(
        actual=('actual', 'sum'), absolute_error=('absolute_error', 'sum')
    ).reindex(ROUTES)
    route_metrics['WAPE'] = (
        route_metrics['absolute_error'] / route_metrics['actual'].replace(0, np.nan)
    )
    return wape, max(0.0, 1.0 - wape), route_metrics.reset_index()

fold_results = []
fold_route_metrics = {}
for fold_id, (train_start, train_end, valid_start, valid_end) in enumerate(FOLDS, start=1):
    fold_train = history.loc[history['date'].between(train_start, train_end)].copy()
    fold_valid = history.loc[history['date'].between(valid_start, valid_end)].copy()
    if fold_train.empty or fold_valid.empty or fold_train['date'].max() >= fold_valid['date'].min():
        raise ValueError(f'Invalid temporal split in fold {fold_id}')
    model = make_model()
    model.fit(
        make_features(fold_train), fold_train['boardings'],
        categorical_feature=list(CATEGORICAL_FEATURES),
    )
    valid_prediction = postprocess_predictions(
        model.predict(make_features(fold_valid))
    )
    wape, wape_score, route_metrics = calculate_metrics(
        fold_valid['boardings'], valid_prediction, fold_valid['route']
    )
    fold_results.append({
        'fold': fold_id,
        'train_period': f'{train_start} — {train_end}',
        'valid_period': f'{valid_start} — {valid_end}',
        'WAPE': wape,
        'WAPE_score': wape_score,
    })
    fold_route_metrics[fold_id] = route_metrics

fold_summary = pd.DataFrame(
    fold_results, columns=['fold', 'train_period', 'valid_period', 'WAPE', 'WAPE_score']
)
display(fold_summary)
for fold_id, route_metrics in fold_route_metrics.items():
    print(f'Fold {fold_id} WAPE by route:')
    display(route_metrics[['route', 'WAPE']].round(6))

,fold,train_period,valid_period,WAPE,WAPE_score
0,1,2025-01-01 — 2025-04-30,2025-05-01 — 2025-06-30,0.184645,0.815355
1,2,2025-01-01 — 2025-06-30,2025-07-01 — 2025-08-31,0.249513,0.750487
2,3,2025-01-01 — 2025-08-31,2025-09-01 — 2025-10-31,0.123793,0.876207


Fold 1 WAPE by route:


,route,WAPE
0,1,0.195197
1,5,NaN
2,7,0.193049
3,11,0.169069
4,12,0.183495
5,17,0.137369
6,25,0.212533
7,26,0.229770
8,28,0.232742
9,50,0.230684


Fold 2 WAPE by route:


,route,WAPE
0,1,0.218437
1,5,NaN
2,7,0.698867
3,11,0.183891
4,12,0.192204
5,17,0.146673
6,25,0.294945
7,26,0.316614
8,28,0.320559
9,50,0.259104


Fold 3 WAPE by route:


,route,WAPE
0,1,0.109409
1,5,NaN
2,7,0.157333
3,11,0.105455
4,12,0.088066
5,17,0.085894
6,25,0.199599
7,26,0.125388
8,28,0.153739
9,50,0.240656


## Final Training

После CV модель обучается на всей полной исторической сетке с января по октябрь 2025. Используются те же признаки, preprocessing и параметры; validation-результаты не используются для подбора.

In [18]:
final_model = make_model()
final_model.fit(
    make_features(history), history['boardings'],
    categorical_feature=list(CATEGORICAL_FEATURES),
)

LGBMRegressor(max_depth=6, n_estimators=300, n_jobs=4,
              objective='regression_l1', random_state=42, verbosity=-1)

## Nov–Dec Inference Grid

Ключи прогноза берутся из официального шаблона и проверяются на полный период 1 ноября — 31 декабря: 10 маршрутов × 61 день × 24 часа. Так сохраняются требуемые ключи и порядок строк шаблона.

In [19]:
inference_grid = submission_template[KEYS].copy()
expected_grid = pd.MultiIndex.from_product(
    [ROUTES, pd.date_range('2025-11-01', '2025-12-31', freq='D'), range(24)],
    names=KEYS,
).to_frame(index=False)
if len(inference_grid) != 14_640 or inference_grid.duplicated(KEYS).any():
    raise ValueError('Submission template has invalid size or duplicate keys')
if not inference_grid.equals(expected_grid):
    raise ValueError('Submission template keys or order differ from expected inference grid')
inference_grid['prediction'] = postprocess_predictions(
    final_model.predict(make_features(inference_grid))
)
if len(inference_grid) != 14_640:
    raise ValueError('Unexpected inference prediction count')

## Submission

Формируется UTF-8 CSV с разделителем `;`, колонками и порядком `route;date;hour;prediction`. Перед записью ключи шаблона сохраняются без переупорядочивания.

In [20]:
submission = inference_grid.loc[:, ['route', 'date', 'hour']].copy()
submission['prediction'] = inference_grid['prediction'].to_numpy(dtype='int64')
submission['date'] = submission['date'].dt.strftime('%Y-%m-%d')
submission = submission.loc[:, ['route', 'date', 'hour', 'prediction']]
if submission.columns.tolist() != ['route', 'date', 'hour', 'prediction']:
    raise ValueError('Submission columns or order are invalid')
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, sep=';', index=False, encoding='utf-8')
print(f'Submission saved to: {SUBMISSION_PATH}')

Submission saved to: /content/drive/MyDrive/MT-Hackathon-2026/data/processed/pilot_model_v1_submission.csv


## Sanity Checks

Проверим формат выгруженного файла, количество строк, диапазоны ключей, уникальность комбинаций и допустимость предсказаний.

In [21]:
submission_check = pd.read_csv(SUBMISSION_PATH, sep=';')
expected_columns = ['route', 'date', 'hour', 'prediction']
if submission_check.columns.tolist() != expected_columns:
    raise ValueError('Submission columns or order are invalid')
if len(submission_check) != 14_640:
    raise ValueError('Submission must contain 14,640 rows')
if submission_check[KEYS].duplicated().any():
    raise ValueError('Submission contains duplicate route/date/hour keys')
if set(submission_check['route'].unique()) != set(ROUTES):
    raise ValueError('Submission route set is invalid')
if submission_check['hour'].nunique() != 24 or not submission_check['hour'].between(0, 23).all():
    raise ValueError('Submission hour values are invalid')
if submission_check['date'].min() != '2025-11-01' or submission_check['date'].max() != '2025-12-31':
    raise ValueError('Submission date range is invalid')
if submission_check['prediction'].isna().any() or (submission_check['prediction'] < 0).any():
    raise ValueError('Submission predictions contain missing or negative values')
if not np.equal(submission_check['prediction'], np.floor(submission_check['prediction'])).all():
    raise ValueError('Submission predictions must be integers')
if submission_check[KEYS].astype(str).values.tolist() != submission_template[KEYS].assign(date=lambda frame: frame['date'].dt.strftime('%Y-%m-%d')).astype(str).values.tolist():
    raise ValueError('Submission keys or row order differ from template')
print('Submission sanity checks passed.')

Submission sanity checks passed.
